# 04 - Perfilamiento e Inferencia
## Segmentacion Inteligente de Pacientes - Programa de Medicina Funcional - Comfama

| | |
|---|---|
| **Proposito** | Construir el modelo de clasificacion (no solo segmentacion) que reemplaza al motor de reglas de la app, entrenarlo, evaluar su viabilidad, guardarlo en data/model.pkl y verificar la integracion con app/domain/ml_model.py. |
| **Entrada** | data/_processed/02_features.csv |
| **Salida** | data/model.pkl - cargado automaticamente por app/domain/classifier.get_classifier(). |

### Por que no basta el clustering conjunto del notebook 03
El notebook 03 responde la pregunta de segmentacion (EDA), pero para
clasificar una consulta individual en Cardiometabolico / Digestivo / Mixto
se necesitan dos cosas que el clustering conjunto de 19 dimensiones no da
limpiamente:

1. Puntajes continuos por eje (para las barras de "afinidad clinica" de la
   app y para poder derivar "Mixto" cuando corresponda).
2. Separacion limpia por eje - la afinidad extraida de un unico clustering
   de 19 dimensiones mezclaba senales (p. ej. el estres alto, sin ningun
   sintoma digestivo, alcanzaba a inflar el puntaje "Digestivo").

La solucion: dos K-Means (k=2) independientes, uno por eje clinico, cada uno
solo con las features relevantes a ese eje.

In [1]:
import sys
from pathlib import Path
import pickle
import numpy as np
import pandas as pd
from scipy.stats import rankdata
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

sys.path.insert(0, str(Path("..").resolve()))
from app.domain.ml_features import FEATURE_NAMES, CARDIO_FEATURES, DIGEST_FEATURES

RANDOM_STATE = 42
MIXTO_PERCENTILE = 0.75

PROCESSED_DIR = Path("../data/_processed")
df = pd.read_csv(PROCESSED_DIR / "02_features.csv")
X = df[FEATURE_NAMES].copy()
medians = X.median(numeric_only=True)
X = X.fillna(medians)

scaler = StandardScaler()
Xs_df = pd.DataFrame(scaler.fit_transform(X), columns=FEATURE_NAMES)
print("Cardio features:", CARDIO_FEATURES)
print("Digest features:", DIGEST_FEATURES)


Cardio features: ['imc', 'perimetro_abdominal', 'pa_sistolica', 'pa_diastolica', 'colesterol_total', 'colesterol_hdl', 'hba1c', 'usa_hipoglicemiante', 'usa_hipolipemiante']
Digest features: ['n_sintomas_gi', 'usa_antiacido_ibp']


### Dos K-Means por eje
DIGEST_FEATURES deliberadamente no incluye estres_alto_bin / tecnica_estres_bin
aunque estan correlacionadas con sintomas digestivos a nivel poblacional (ver
perfil del notebook 03): al ser binarias con baja prevalencia, dominaban la
distancia de un espacio de solo 4 dimensiones y producian falsos "Digestivo"
en pacientes con estres alto pero sin ningun sintoma gastrointestinal. Si se
usan en el clustering conjunto (EDA), no en el puntaje de clasificacion.

In [2]:
Xc = Xs_df[CARDIO_FEATURES].values
Xd = Xs_df[DIGEST_FEATURES].values

km_cardio = KMeans(n_clusters=2, random_state=RANDOM_STATE, n_init=10).fit(Xc)
km_digest = KMeans(n_clusters=2, random_state=RANDOM_STATE, n_init=10).fit(Xd)

sil_cardio = silhouette_score(Xc, km_cardio.labels_, sample_size=5000, random_state=RANDOM_STATE)
sil_digest = silhouette_score(Xd, km_digest.labels_, sample_size=5000, random_state=RANDOM_STATE)
print(f"Silueta eje cardiometabolico (k=2): {sil_cardio:.4f}")
print(f"Silueta eje digestivo (k=2):        {sil_digest:.4f}")
print("(ambas muy por encima de la silueta 0.144 del clustering conjunto de 19 dimensiones)")

hi_cardio_cluster = int(df.groupby(km_cardio.labels_)["hba1c"].mean().idxmax())
hi_digest_cluster = int(df.groupby(km_digest.labels_)["n_sintomas_gi"].mean().idxmax())
print(f"\ncluster de alto riesgo cardio: {hi_cardio_cluster} | digestivo: {hi_digest_cluster}")


Silueta eje cardiometabolico (k=2): 0.3184
Silueta eje digestivo (k=2):        0.8317
(ambas muy por encima de la silueta 0.144 del clustering conjunto de 19 dimensiones)

cluster de alto riesgo cardio: 1 | digestivo: 1


### Puntaje de afinidad y regla de decision
La afinidad de un paciente a un eje es su percentil de cercania al centroide
de "alto riesgo" de ese eje, respecto a la distribucion de distancias del
historico. Si ambos puntajes caen en el cuartil superior (percentil mayor o
igual a 0.75), el fenotipo es Mixto; si no, gana el eje con mayor percentil.
Este umbral reemplaza los pesos/umbrales manuales del antiguo clasificador
de reglas por valores aprendidos de los datos, conservando la misma idea
(dos puntajes + una regla de decision) para no romper la interfaz Classifier
ni la UI existente.

In [3]:
dist_cardio = np.linalg.norm(Xc - km_cardio.cluster_centers_[hi_cardio_cluster], axis=1)
dist_digest = np.linalg.norm(Xd - km_digest.cluster_centers_[hi_digest_cluster], axis=1)

pct_cardio = 1 - (rankdata(dist_cardio) / len(dist_cardio))
pct_digest = 1 - (rankdata(dist_digest) / len(dist_digest))

final_label = np.where(
    (pct_cardio >= MIXTO_PERCENTILE) & (pct_digest >= MIXTO_PERCENTILE), "Mixto",
    np.where(pct_cardio >= pct_digest, "Cardiometabólico", "Digestivo"),
)
print(pd.Series(final_label).value_counts())
print(pd.Series(final_label).value_counts(normalize=True).round(3))


Cardiometabólico    8157
Digestivo           7649
Mixto                368
Name: count, dtype: int64
Cardiometabólico    0.504
Digestivo           0.473
Mixto               0.023
Name: proportion, dtype: float64


In [4]:
df.assign(fen=final_label).groupby("fen")[FEATURE_NAMES].mean().round(2).T


fen,Cardiometabólico,Digestivo,Mixto
imc,29.43,27.25,29.19
perimetro_abdominal,94.31,86.87,95.08
pa_sistolica,119.04,117.55,119.09
pa_diastolica,77.76,76.15,78.49
colesterol_total,186.43,200.69,181.16
colesterol_hdl,46.56,52.26,46.26
hba1c,5.70,5.46,5.83
usa_hipoglicemiante,0.25,0.03,0.05
usa_hipolipemiante,0.42,0.09,0.08
n_sintomas_gi,0.31,0.90,1.90


Validacion clinica del resultado: el grupo "Cardiometabolico" tiene el IMC,
perimetro abdominal, presion arterial, HbA1c y uso de medicacion mas altos;
el grupo "Digestivo" tiene mas del doble de sintomas gastrointestinales
promedio y mayor uso de antiacidos/IBP; el grupo "Mixto" (~2.3% del
historico) tiene ambos ejes elevados simultaneamente - mas sintomas
digestivos que el propio grupo Digestivo, y marcadores cardiometabolicos
cercanos al grupo Cardiometabolico. Es una proporcion pequena pero
clinicamente sensata, coherente con el hallazgo del notebook 03 (no hay un
cluster natural "ambos ejes altos").

### Guardado del modelo
El bundle incluye todo lo necesario para inferencia: el escalador, las
medianas de imputacion, los dos K-Means por eje, cual cluster es "de riesgo"
en cada uno, y las distancias de entrenamiento (para el percentil en
produccion).

In [5]:
MODEL_PATH = Path("../data/model.pkl")
bundle = {
    "version": 2,
    "feature_names": FEATURE_NAMES,
    "cardio_features": CARDIO_FEATURES,
    "digest_features": DIGEST_FEATURES,
    "medians": medians.to_dict(),
    "scaler": scaler,
    "km_cardio": km_cardio,
    "km_digest": km_digest,
    "hi_cardio_cluster": hi_cardio_cluster,
    "hi_digest_cluster": hi_digest_cluster,
    "dist_cardio_train": dist_cardio,
    "dist_digest_train": dist_digest,
    "mixto_percentile": MIXTO_PERCENTILE,
}
with open(MODEL_PATH, "wb") as fh:
    pickle.dump(bundle, fh)
print(f"Modelo guardado en {MODEL_PATH.resolve()}")


Modelo guardado en C:\hackaton2026\FenotipoMedico\data\model.pkl


### Verificacion de la integracion con la app
app/domain/classifier.get_classifier() ya detecta data/model.pkl y devuelve
TrainedClassifier (definido en app/domain/ml_model.py) automaticamente, sin
tocar la UI. Se prueba aqui con dos perfiles sinteticos.

In [6]:
from app.domain.classifier import get_classifier

clf = get_classifier()
print("Clasificador activo:", type(clf).__name__)

perfil_cardio = {
    "sexo": "Masculino", "medicamentos": ["Hipoglicemiantes (para el azúcar)", "Hipolipemiantes (para el colesterol)"],
    "sintomas_digestivos": ["Ninguno"], "altas_cargas_estres": False, "tecnica_gestion_estres": False,
    "bmi": 33.0, "waist_cm": 110, "bp_systolic": 145, "bp_diastolic": 92,
    "colesterol_total": 240, "hdl": 38, "hba1c": 7.2,
}
r = clf.predict(perfil_cardio)
print("\nPerfil cardiometabólico ->", r.phenotype, r.scores)

perfil_digest = {
    "sexo": "Femenino", "medicamentos": ["Inhibidores de bomba de protones / antiácidos (omeprazol, etc.)"],
    "sintomas_digestivos": ["Reflujo gastroesofágico", "Gases", "Distensión abdominal", "Sensación de agriera"],
    "altas_cargas_estres": True, "tecnica_gestion_estres": False,
    "bmi": 23.0, "waist_cm": 70, "bp_systolic": 110, "bp_diastolic": 70,
    "colesterol_total": 170, "hdl": 60, "hba1c": 5.0,
}
r2 = clf.predict(perfil_digest)
print("Perfil digestivo         ->", r2.phenotype, r2.scores)


Clasificador activo: TrainedClassifier

Perfil cardiometabólico -> Cardiometabólico {'Cardiometabólico': 0.216, 'Digestivo': 0.025}
Perfil digestivo         -> Digestivo {'Cardiometabólico': 0.109, 'Digestivo': 0.945}


### Comparacion con el clasificador de reglas anterior
Para evaluar viabilidad, se compara la clasificacion del modelo entrenado
contra el motor de reglas original sobre un conjunto de perfiles sinteticos
representativos (no sobre el propio historico de entrenamiento, que no tiene
etiqueta real de fenotipo - este es un problema no supervisado, no hay
"ground truth" contra la cual medir accuracy clasica).

In [7]:
from app.domain.classifier import RuleBasedClassifier

rule_clf = RuleBasedClassifier()
casos = [
    ("Cardiometabólico típico", perfil_cardio),
    ("Digestivo típico", perfil_digest),
    ("Sin datos clínicos (solo cuestionario)", {
        "sexo": "Femenino", "medicamentos": ["Ninguno"], "sintomas_digestivos": ["Ninguno"],
        "altas_cargas_estres": True, "tecnica_gestion_estres": False,
    }),
]
for nombre, features in casos:
    r_regla = rule_clf.predict(features)
    r_modelo = clf.predict(features)
    print(f"{nombre:45s} | reglas: {r_regla.phenotype:18s} | modelo: {r_modelo.phenotype}")


Cardiometabólico típico                       | reglas: Cardiometabólico   | modelo: Cardiometabólico
Digestivo típico                              | reglas: Digestivo          | modelo: Digestivo
Sin datos clínicos (solo cuestionario)        | reglas: Cardiometabólico   | modelo: Digestivo


## Conclusiones y viabilidad

Es viable el modelo para clasificar pacientes? Si, con matices:

- Sobre perfiles clinicos claros (marcadores cardiometabolicos vs. sintomas
  digestivos evidentes) el modelo coincide con el criterio clinico esperado
  y con el clasificador de reglas anterior.
- La separacion por eje es fuerte: silueta 0.32 (cardiometabolico) y 0.83
  (digestivo) para los K-Means de 2 clusters - mucho mas limpia que
  cualquier intento de extraerla de un unico clustering conjunto.
- Maneja con gracia la falta de datos clinicos (muy comun: un paciente
  recien llegado del chat, antes de su primera cita medica): si a un eje le
  falta toda la informacion numerica, devuelve un puntaje neutral en vez de
  uno distorsionado por imputacion.
- Limitacion 1 - sin "Mixto" natural. El EDA (notebook 03) muestra que no
  existe un cluster propio para "ambos ejes elevados"; "Mixto" se deriva con
  una regla de umbral (percentil mayor o igual a 0.75 en ambos ejes) en vez
  de salir directamente del clustering. Es razonable pero es una decision de
  diseno, no un hallazgo puramente no supervisado.
- Limitacion 2 - sin variable de sexo en el historico. El Excel no trae sexo
  biologico, asi que el modelo no ajusta el perimetro abdominal por sexo (el
  clasificador de reglas anterior si lo hacia). El perimetro entra al modelo
  sin ese ajuste; la explicacion en lenguaje sencillo (rationale) si usa el
  sexo de la app cuando esta disponible.
- Limitacion 3 - sin "ground truth". Al ser no supervisado, la validacion es
  por coherencia clinica de los perfiles (rangos de IMC, presion, HbA1c,
  sintomas) y por consistencia con el motor de reglas, no por accuracy
  contra una etiqueta real. Se recomienda que el equipo medico revise una
  muestra de clasificaciones del modelo (via la reclasificacion manual ya
  disponible en la consola medica) para ir acumulando esa validacion con el
  tiempo.

Categorias definidas a partir del EDA: los datos confirman las 3 categorias
de negocio ya usadas por la app - Cardiometabolico, Digestivo y Mixto - con
el matiz de que "Mixto" es una condicion derivada (ambos puntajes altos) y
no un cluster natural en si mismo; el verdadero tercer segmento que emerge
del clustering es mas bien "bajo riesgo / sin eje predominante", que el
modelo resuelve asignando al eje con mayor afinidad relativa (igual que
hacia el clasificador de reglas para casos de senal debil).